# Tarea 3 — PyTorch: Fashion MNIST

**Nombre del estudiante:** Anthony Caldera  
**Cédula de Identidad:** 30473999  

---
### Descripción general
Este programa implementa redes neuronales totalmente conectadas utilizando **PyTorch** para clasificar el conjunto de datos **Fashion MNIST** tanto en su formato original (10 clases) como en el formato simplificado (4 clases).

Permite configurar arquitecturas **rectangulares** y **convolucionales** (embudo por interpolación lineal), aplicar optimizaciones (Batch Normalization, Dropout, Weight Decay), entrenar evaluando la función de pérdida por época, medir el desempeño mediante matriz de confusión y métricas macro (Accuracy, Precision, Recall), y persistir los modelos bajo el contrato requerido (`*.pth`).

## 1. Imports y constantes globales

In [ ]:
# --- Standard and Third-Party Libraries ---
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
)

# --- Device Selection ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Usando dispositivo: {device}")

# --- Class Definitions and Label Mappings ---
ORIGINAL_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

SIMPLIFIED_CLASSES = ["Top", "Bottom", "Footwear", "Bag"]

# Fashion MNIST -> Simplified category mapping:
# 0: Top      (T-shirt: 0, Pullover: 2, Coat: 4, Shirt: 6)
# 1: Bottom   (Trouser: 1, Dress: 3)
# 2: Footwear (Sandal: 5, Sneaker: 7, Ankle boot: 9)
# 3: Bag      (Bag: 8)
SIMPLIFIED_LABEL_MAP = {
    0: 0, 1: 1, 2: 0, 3: 1, 4: 0,
    5: 2, 6: 0, 7: 2, 8: 3, 9: 2
}


## 2. Carga y preparación de datos

In [ ]:
def get_fashion_mnist_loaders(batch_size=64, data_dir="./data"):
    """
    Descarga Fashion MNIST (si no existe) y retorna los DataLoaders de entrenamiento y prueba.
    Los pixeles se transforman a tensor en rango [0, 1] sin normalizaciones externas.
    """
    transform = transforms.ToTensor()
    train_set = torchvision.datasets.FashionMNIST(
        root=data_dir, train=True, download=True, transform=transform
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=data_dir, train=False, download=True, transform=transform
    )
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

def transform_targets(targets, classification_type):
    """
    Transforma las etiquetas de las 10 clases originales a las 4 clases simplificadas
    cuando classification_type == 'simplified'.
    """
    if classification_type == "simplified":
        return torch.tensor([SIMPLIFIED_LABEL_MAP[val.item()] for val in targets], device=targets.device)
    return targets


## 3. Construcción del modelo (Rectangular y Convolucional/Embudo)

In [ ]:
def calculate_convolutional_neurons(num_layers, out_features, in_features=784):
    """
    Calcula el número de neuronas por capa oculta usando la fórmula de interpolación lineal:
    neurons(i) = round(in_features + (out_features - in_features) * i / (num_layers + 1))
    """
    return [
        round(in_features + (out_features - in_features) * i / (num_layers + 1))
        for i in range(1, num_layers + 1)
    ]

def build_model(hyperparams):
    """
    Construye un modelo nn.Sequential que cumple con el contrato estricto del enunciado:
    - Inicia con nn.Flatten()
    - Solo utiliza nn.Linear, nn.ReLU, nn.BatchNorm1d y nn.Dropout
    - Entrada: 784 píxeles
    - Salida: 10 neuronas (original) o 4 neuronas (simplificada)
    """
    net_type = hyperparams.get("net_type", "rectangular")
    num_layers = hyperparams.get("hidden_layers", 2)
    classification = hyperparams.get("classification", "original")
    use_batchnorm = hyperparams.get("batchnorm", False)
    use_dropout = hyperparams.get("dropout", False)
    dropout_rate = hyperparams.get("dropout_rate", 0.2)
    
    in_features = 784
    out_features = 10 if classification == "original" else 4
    
    # Determinar la lista de neuronas para cada capa oculta
    if net_type == "rectangular":
        neurons_per_layer = hyperparams.get("neurons_per_layer", 512)
        hidden_sizes = [neurons_per_layer] * num_layers
    else:
        # Convolucional / embudo por interpolación lineal
        hidden_sizes = calculate_convolutional_neurons(num_layers, out_features, in_features)
    
    # Ensamblar el Sequential
    layers = [nn.Flatten()]
    current_in = in_features
    
    for hidden_size in hidden_sizes:
        layers.append(nn.Linear(current_in, hidden_size))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(hidden_size))
        layers.append(nn.ReLU())
        if use_dropout:
            layers.append(nn.Dropout(dropout_rate))
        current_in = hidden_size
    
    # Capa de salida
    layers.append(nn.Linear(current_in, out_features))
    
    model = nn.Sequential(*layers)
    return model.to(device)


## 4. Funciones de Entrenamiento, Evaluación y Persistencia

In [ ]:
def train_network(model, train_loader, epochs, lr, optimizer_choice="adam", classification="original", weight_decay=0.0):
    """
    Entrena la red por un número determinado de épocas y grafica la pérdida media por época.
    """
    criterion = nn.CrossEntropyLoss()
    if optimizer_choice.lower() == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        
    model.train()
    history_loss = []
    
    print(f"\n[ENTRENAMIENTO] Iniciando por {epochs} épocas (lr={lr}, optimizador={optimizer_choice})...")
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        for images, targets in train_loader:
            images = images.to(device)
            targets = transform_targets(targets.to(device), classification)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_loss = running_loss / len(train_loader)
        history_loss.append(epoch_loss)
        print(f"  Época [{epoch}/{epochs}] - Loss: {epoch_loss:.4f}")
        
    # Gráfico de pérdida vs épocas
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, epochs + 1), history_loss, marker='o', color='teal', linewidth=2)
    plt.title("Pérdida (Loss) vs Épocas de Entrenamiento")
    plt.xlabel("Época")
    plt.ylabel("Cross Entropy Loss")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()
    
    return history_loss

def evaluate_network(model, test_loader, classification="original"):
    """
    Evalúa el modelo contra el conjunto de prueba, calcula métricas macro y dibuja la matriz de confusión.
    """
    model.eval()
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for images, targets in test_loader:
            images = images.to(device)
            targets = transform_targets(targets.to(device), classification)
            
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            
            y_true.extend(targets.cpu().numpy())
            y_pred.extend(predictions.cpu().numpy())
            
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    
    print("\n" + "="*40)
    print("          RESULTADOS DE LA PRUEBA")
    print("="*40)
    print(f"  Accuracy:   {acc * 100:.2f}%")
    print(f"  Precision:  {prec * 100:.2f}% (Macro)")
    print(f"  Recall:     {rec * 100:.2f}% (Macro)")
    print("="*40)
    
    class_names = ORIGINAL_CLASSES if classification == "original" else SIMPLIFIED_CLASSES
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
    plt.title(f"Matriz de Confusión ({classification.capitalize()})")
    plt.tight_layout()
    plt.show()
    
    return acc, prec, rec, cm

def save_checkpoint(model, hyperparams, filepath):
    """
    Guarda el modelo cumpliendo estrictamente con el contrato del enunciado:
    torch.save({'state_dict': model.state_dict(), 'hyperparameters': {...}}, path)
    """
    data = {
        "state_dict": model.state_dict(),
        "hyperparameters": hyperparams
    }
    torch.save(data, filepath)
    print(f"\n[EXITO] Red neuronal guardada exitosamente en: {filepath}")

def load_checkpoint(filepath):
    """
    Carga un archivo .pth, reconstruye la arquitectura a partir de sus hiperparámetros
    y carga su state_dict.
    """
    data = torch.load(filepath, map_location=device, weights_only=False)
    hyperparams = data["hyperparameters"]
    model = build_model(hyperparams)
    model.load_state_dict(data["state_dict"])
    print(f"\n[EXITO] Red neuronal cargada exitosamente desde: {filepath}")
    return model, hyperparams


## 5. Menú interactivo

El menú principal permite crear una nueva red o cargar una existente, entrenarla repetidamente visualizando su pérdida, probarla mostrando la matriz de confusión y métricas, y guardar el modelo en disco.

In [ ]:
def prompt_int(message, default_val):
    val = input(f"{message} [por defecto {default_val}]: ").strip()
    return int(val) if val else default_val

def prompt_float(message, default_val):
    val = input(f"{message} [por defecto {default_val}]: ").strip()
    return float(val) if val else default_val

def prompt_bool(message, default_val=False):
    default_str = "s" if default_val else "n"
    val = input(f"{message} (s/n) [por defecto {default_str}]: ").strip().lower()
    if not val:
        return default_val
    return val in ["s", "si", "y", "yes", "true", "1"]

def run_interactive_app():
    """
    Flujo principal de la aplicación con menú interactivo.
    """
    train_loader, test_loader = get_fashion_mnist_loaders(batch_size=64)
    model = None
    hyperparams = None
    
    print("="*50)
    print("       CLASIFICADOR PYTORCH FASHION MNIST")
    print("               Anthony Caldera (30473999)")
    print("="*50)
    print("1. Crear una nueva red neuronal")
    print("2. Cargar una red desde archivo (.pth)")
    start_choice = input("Seleccione una opción (1/2): ").strip()
    
    if start_choice == "1":
        print("\n--- CONFIGURACIÓN INICIAL DE LA RED ---")
        task_opt = input("Clasificación (1: Original 10 clases, 2: Simplificada 4 clases) [por defecto 1]: ").strip()
        classification = "simplified" if task_opt == "2" else "original"
        
        type_opt = input("Tipo de arquitectura (1: Rectangular, 2: Convolucional) [por defecto 1]: ").strip()
        net_type = "convolucional" if type_opt == "2" else "rectangular"
        
        hidden_layers = prompt_int("Número de capas ocultas", 2)
        
        hyperparams = {
            "classification": classification,
            "net_type": net_type,
            "hidden_layers": hidden_layers,
            "accumulated_epochs": 0
        }
        
        if net_type == "rectangular":
            hyperparams["neurons_per_layer"] = prompt_int("Número de neuronas por capa oculta", 512)
            
        print("\n--- OPTIMIZACIONES DISPONIBLES ---")
        hyperparams["batchnorm"] = prompt_bool("¿Habilitar Batch Normalization?", False)
        hyperparams["dropout"] = prompt_bool("¿Habilitar Dropout?", False)
        if hyperparams["dropout"]:
            hyperparams["dropout_rate"] = prompt_float("Porcentaje de Dropout (ej. 0.2)", 0.2)
            
        model = build_model(hyperparams)
        print("\n[EXITO] Red neuronal creada correctamente.")
        print(model)
        
    elif start_choice == "2":
        filepath = input("Ingrese la ruta del archivo .pth a cargar: ").strip()
        if not os.path.exists(filepath):
            print(f"[ERROR] El archivo '{filepath}' no existe.")
            return
        model, hyperparams = load_checkpoint(filepath)
    else:
        print("[INFO] Opción no válida. Saliendo del programa.")
        return
        
    # Bucle del Menú de Operaciones
    while True:
        print("\n" + "-"*45)
        print("                MENÚ DE ACCIONES")
        print("-"*45)
        print(f"  Tipo de Red:       {hyperparams.get('net_type', '').capitalize()}")
        print(f"  Clasificación:     {hyperparams.get('classification', '')}")
        print(f"  Capas Ocultas:     {hyperparams.get('hidden_layers', 0)}")
        print(f"  Épocas acumuladas: {hyperparams.get('accumulated_epochs', 0)}")
        print("-"*45)
        print("  1. Entrenar la red")
        print("  2. Probar la red (matriz de confusión y métricas)")
        print("  3. Guardar la red (*.pth)")
        print("  4. Salir")
        print("-"*45)
        
        choice = input("Seleccione una acción (1-4): ").strip()
        
        if choice == "1":
            epochs = prompt_int("Número de épocas a entrenar", 10)
            lr = prompt_float("Learning Rate", 0.001)
            opt_sel = input("Optimizador (1: Adam, 2: SGD) [por defecto 1]: ").strip()
            optimizer_choice = "sgd" if opt_sel == "2" else "adam"
            weight_decay = prompt_float("Weight decay (L2 regularización)", 0.0)
            
            train_network(
                model=model,
                train_loader=train_loader,
                epochs=epochs,
                lr=lr,
                optimizer_choice=optimizer_choice,
                classification=hyperparams["classification"],
                weight_decay=weight_decay
            )
            hyperparams["accumulated_epochs"] = hyperparams.get("accumulated_epochs", 0) + epochs
            
        elif choice == "2":
            print("\n[INFO] Evaluando la red sobre el conjunto de prueba...")
            evaluate_network(model, test_loader, classification=hyperparams["classification"])
            
        elif choice == "3":
            suggested_name = f"pytorch_fashion_2_{hyperparams['classification']}_{hyperparams['net_type']}.pth"
            filepath = input(f"Nombre de archivo para guardar [por defecto: {suggested_name}]: ").strip()
            if not filepath:
                filepath = suggested_name
            save_checkpoint(model, hyperparams, filepath)
            
        elif choice == "4":
            print("\n[INFO] Finalizando la sesión. ¡Hasta luego!")
            break
        else:
            print("[AVISO] Opción inválida. Por favor seleccione un número del 1 al 4.")

# Ejecutar aplicación
run_interactive_app()
